# Voltage pipeline results

- `mbo voltage <expt>.mesc` writes a `PF` folder next to the experiment.
- `PF/traces/` holds the plain outputs below. Everything else in `PF` is the archive format that `mbo curate` opens.

| file | contents |
|---|---|
| `scans.csv` | one row per scan: id, MESc unit, frame rate, frames, ROIs |
| `domains.csv` | one row per domain: row index, name, ROI indices |
| `scan<id>_rois.npy` | raw mean fluorescence, shape (ROI, frame) |
| `scan<id>_dfof.npy` | dF/F per domain, shape (domain, frame) |
| `scan<id>_zscore.npy` | z-scored dF/F per domain, shape (domain, frame) |
| `scan<id>_denoised.npy` | wavelet-denoised trace per domain, shape (domain, frame) |
| `scan<id>_peaks.csv` | detected events: domain, frame, time in seconds |
| `scan<id>_denoised.png`, `scan<id>_rois.png` | figures of the above |
| `../.curation/fast_template_curation.json` | events labelled yes / no in the curation window |

- Row `i` of every `(domain, frame)` array is row `i` of `domains.csv`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

traces = Path(r"X:/data/asako/stan112/stan112_expt12/PF") / "traces"

## Scans

In [ ]:
scans = pd.read_csv(traces / "scans.csv")
scans

## Domains

In [ ]:
domains = pd.read_csv(traces / "domains.csv")
domains

## Traces

In [ ]:
scan = str(scans.scan[0])
rois = np.load(traces / f"scan{scan}_rois.npy")
dfof = np.load(traces / f"scan{scan}_dfof.npy")
zscore = np.load(traces / f"scan{scan}_zscore.npy")
denoised = np.load(traces / f"scan{scan}_denoised.npy")
pd.DataFrame({"array": ["rois", "dfof", "zscore", "denoised"], "shape": [rois.shape, dfof.shape, zscore.shape, denoised.shape]})

- One domain as a table: time in seconds, dF/F, z-score, denoised.

In [ ]:
fs = float(scans.fs_hz[0])
row = 0
pd.DataFrame({"time_s": np.arange(dfof.shape[1]) / fs, "dfof": dfof[row], "zscore": zscore[row], "denoised": denoised[row]}).head()

## Events

In [ ]:
peaks = pd.read_csv(traces / f"scan{scan}_peaks.csv")
peaks.groupby("domain").size().rename("n_events").to_frame()

In [ ]:
peaks.head()

## Figures

In [ ]:
from IPython.display import Image

Image(traces / f"scan{scan}_denoised.png")

In [ ]:
Image(traces / f"scan{scan}_rois.png")

## Curated events

- Written by the curation window (`mbo <expt>.mesc`, Curation tab); empty until events are labelled.

In [ ]:
import json

labels = traces.parent / ".curation" / "fast_template_curation.json"
events = json.loads(labels.read_text())["events"] if labels.exists() else {}
curated = pd.DataFrame([{"recording": e["recording"], "frame": e["source_event_index"], "time_s": e["source_event_time_s"], "label": e["label"]} for e in events.values()])
curated